# PROCESAMIENTO DIVIPOLA

## 1. LIBRERÍAS Y RUTAS

In [1]:
import pandas as pd
import numpy as np

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_ENTRADA = (
    "/kaggle/input/datasets/nicolasacostaa/divipola-colombia/"
    "DIVIPOLA-_Cdigos_municipios_geolocalizados_20260411.csv"
)
RUTA_SALIDA = "/kaggle/working/divipola_geo_procesado.parquet"

# ── Mapeo de nombres de columnas de salida ────────────────────────────────────
RENOMBRAR = {
    "COD_DPTO"     : "cod_dpto",
    "NOM_DPTO"     : "nom_dpto",
    "COD_MPIO"     : "cod_mpio",
    "NOM_MPIO"     : "nom_mpio",
    "TIPO"         : "tipo",
    "LATITUD"      : "latitud",
    "LONGITUD"     : "longitud",
    "Geo Municipio": "geometry_wkt",
}

## 2. FUNCIONES

### 2.1. Cargar divipola

In [2]:
def cargar_divipola(ruta: str) -> pd.DataFrame:
    """
    Carga el CSV de DIVIPOLA geolocalizados.

    El archivo tiene las siguientes particularidades que se resuelven
    en las funciones de transformación posteriores:
      - COD_DPTO  : int64, puede tener 1 o 2 dígitos (ej. 5 → '05').
      - COD_MPIO  : float64 con el municipio codificado como decimal
                    (ej. 5.030 representa dpto=05, mpio=030).
      - LATITUD   : string con coma como separador decimal.
      - LONGITUD  : string con coma como separador decimal.

    Parameters
    ----------
    ruta : str
        Ruta al CSV dentro del entorno de Kaggle.

    Returns
    -------
    pd.DataFrame
        DataFrame crudo con los 8 campos originales.
    """
    print(f"[INFO] Cargando DIVIPOLA desde: {ruta}")
    df = pd.read_csv(ruta, encoding="utf-8")
    print(f"  → Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    print(f"  → Columnas: {df.columns.tolist()}")
    return df

### 2.2. Procesar Códigos

In [3]:
def procesar_codigos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera las claves de cruce con el dataset de defunciones:

    COD_DPTO (procesado)
    ─────────────────────
    El campo original es un entero de 1 o 2 dígitos.
    Se convierte a string de 2 dígitos con cero a la izquierda.
    Ejemplo: 5 → '05'  |  91 → '91'

    COD_MPIO (procesado)
    ─────────────────────
    El campo original es un float donde:
      - La parte entera coincide con COD_DPTO.
      - La parte decimal codifica el municipio en 3 dígitos.
    Ejemplo: 5.030 → parte entera=5, decimal × 1000 = 30 → '030'
             → COD_MPIO final = '05' + '030' = '05030'

    El uso de round() antes de convertir a int evita errores de
    precisión de punto flotante (ej. 5.030 × 1000 = 29.9999...).

    Compatibilidad:
      - COD_DPTO cruza con 'CÓDIGO DEPARTAMENTO' en defunciones.
      - COD_MPIO cruza con 'cod_muni_def' en defunciones.

    Parameters
    ----------
    df : pd.DataFrame   DataFrame crudo cargado por cargar_divipola().

    Returns
    -------
    pd.DataFrame
        DataFrame con COD_DPTO y COD_MPIO reemplazados por sus versiones
        procesadas como strings.
    """
    print("[INFO] Procesando COD_DPTO y COD_MPIO...")
    df = df.copy()

    # COD_DPTO → string de 2 dígitos
    df["COD_DPTO"] = df["COD_DPTO"].astype(int).astype(str).str.zfill(2)

    # Parte municipal: (float % 1) × 1000 → entero → zfill(3)
    parte_mpio = (
        df["COD_MPIO"]
        .pipe(lambda s: (s % 1) * 1000)   # extraer decimales escalados
        .round(0)                          # corregir imprecisión float
        .astype(int)
        .astype(str)
        .str.zfill(3)
    )

    # COD_MPIO final = dept (2 dígitos) + mpio (3 dígitos)
    df["COD_MPIO"] = df["COD_DPTO"] + parte_mpio

    print(f"  → COD_DPTO muestra : {df['COD_DPTO'].head(5).tolist()}")
    print(f"  → COD_MPIO muestra : {df['COD_MPIO'].head(5).tolist()}")
    print(f"  → Municipios únicos: {df['COD_MPIO'].nunique():,}")
    return df

### 2.3. Procesar Coordenadas

In [4]:
def procesar_coordenadas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte LATITUD y LONGITUD de string con coma decimal a float64.

    El CSV original usa la coma como separador decimal en estas columnas
    (formato europeo), por lo que pandas las carga como strings.
    Ejemplo: '6,257590259' → 6.257590259

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Convirtiendo LATITUD y LONGITUD a float64...")
    df = df.copy()

    for col in ["LATITUD", "LONGITUD"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
        n_nulos = df[col].isna().sum()
        print(f"  → {col}: rango [{df[col].min():.4f}, {df[col].max():.4f}]  "
              f"| nulos: {n_nulos}")

    return df

### 2.4. Limpiar Campos Texto

In [5]:
def limpiar_campos_texto(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica limpieza estándar a todos los campos de texto:
      - Elimina espacios iniciales y finales.
      - Normaliza a mayúsculas los campos de nombre y tipo.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Limpiando campos de texto...")
    df = df.copy()

    cols_upper = ["NOM_DPTO", "NOM_MPIO", "TIPO"]
    for col in cols_upper:
        df[col] = df[col].astype(str).str.strip().str.upper()

    # geometry_wkt: solo strip, conservar el formato WKT original
    df["Geo Municipio"] = df["Geo Municipio"].astype(str).str.strip()

    print(f"  → TIPO valores únicos : {sorted(df['TIPO'].unique())}")
    return df

### 2.5. Renombrar y Ordenar

In [6]:
def renombrar_y_ordenar(df: pd.DataFrame, renombrar: dict) -> pd.DataFrame:
    """
    Renombra las columnas al estándar en minúsculas y establece el
    orden final del DataFrame.

    Orden de salida:
      cod_dpto | nom_dpto | cod_mpio | nom_mpio | tipo |
      latitud  | longitud | geometry_wkt

    Parameters
    ----------
    df : pd.DataFrame
    renombrar : dict   Mapeo nombre_original → nombre_final.

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Renombrando y ordenando columnas...")
    df = df.rename(columns=renombrar)
    orden = ["cod_dpto", "nom_dpto", "cod_mpio", "nom_mpio",
             "tipo", "latitud", "longitud", "geometry_wkt"]
    df = df[orden]
    print(f"  → Columnas finales: {df.columns.tolist()}")
    return df

### 2.6. Diagnosticar

In [7]:
def diagnosticar(df: pd.DataFrame) -> None:
    """
    Imprime un resumen de calidad del DataFrame procesado:
      - Shape y tipos de datos.
      - Valores nulos por columna.
      - Muestra de filas representativas.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    None
    """
    print("\n[INFO] ── Diagnóstico del DataFrame procesado ──")
    print(f"  → Shape      : {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  → Deptos únicos : {df['cod_dpto'].nunique()}")
    print(f"  → Mpios únicos  : {df['cod_mpio'].nunique():,}")
    print()
    print("  Tipos de datos:")
    for col, dtype in df.dtypes.items():
        nulos = df[col].isna().sum()
        print(f"    {col:<20} {str(dtype):<12} nulos: {nulos}")
    print()
    print("  Muestra (primeras 5 filas):")
    display(df.head(5))
    print()
    print("  Muestra (deptos especiales — Bogotá, Amazonas, San Andrés):")
    display(df[df["cod_dpto"].isin(["11", "91", "88"])].head(6))

### 2.7. Exportar parquet

In [8]:
def exportar_parquet(df: pd.DataFrame,
                     ruta_salida: str,
                     compresion: str = "snappy") -> None:
    """
    Exporta el DataFrame procesado a formato Parquet.

    Parameters
    ----------
    df : pd.DataFrame
    ruta_salida : str   Ruta de destino incluyendo nombre y extensión.
    compresion : str    'snappy' (default), 'gzip' o 'brotli'.

    Returns
    -------
    None
    """
    print(f"[INFO] Exportando a Parquet: {ruta_salida}")
    df.to_parquet(ruta_salida, index=False, compression=compresion)
    print(f"  → Guardado | Filas: {len(df):,} | Columnas: {df.shape[1]}")

## 3. Ejecutar

In [9]:
# ── Carga ─────────────────────────────────────────────────────────────────────
df_raw = cargar_divipola(RUTA_ENTRADA)

# ── Transformaciones ──────────────────────────────────────────────────────────
df = (
    df_raw
    .pipe(procesar_codigos)        # COD_DPTO zfill(2) | COD_MPIO 5 dígitos
    .pipe(procesar_coordenadas)    # LATITUD / LONGITUD → float64
    .pipe(limpiar_campos_texto)    # NOM_DPTO, NOM_MPIO, TIPO → upper + strip
    .pipe(renombrar_y_ordenar, RENOMBRAR)  # nombres en minúsculas + orden
)

# ── Diagnóstico ───────────────────────────────────────────────────────────────
diagnosticar(df)

# ── Exportar ──────────────────────────────────────────────────────────────────
exportar_parquet(df, RUTA_SALIDA)

print("\n[OK] Pipeline completado.")

[INFO] Cargando DIVIPOLA desde: /kaggle/input/datasets/nicolasacostaa/divipola-colombia/DIVIPOLA-_Cdigos_municipios_geolocalizados_20260411.csv
  → Filas: 1,121  |  Columnas: 8
  → Columnas: ['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO', 'TIPO', 'LATITUD', 'LONGITUD', 'Geo Municipio']
[INFO] Procesando COD_DPTO y COD_MPIO...
  → COD_DPTO muestra : ['05', '05', '05', '05', '05']
  → COD_MPIO muestra : ['05001', '05002', '05004', '05021', '05030']
  → Municipios únicos: 1,121
[INFO] Convirtiendo LATITUD y LONGITUD a float64...
  → LATITUD: rango [-3.6313, 13.3511]  | nulos: 0
  → LONGITUD: rango [-81.7176, -67.0015]  | nulos: 0
[INFO] Limpiando campos de texto...
  → TIPO valores únicos : ['ISLA', 'MUNICIPIO', 'ÁREA NO MUNICIPALIZADA']
[INFO] Renombrando y ordenando columnas...
  → Columnas finales: ['cod_dpto', 'nom_dpto', 'cod_mpio', 'nom_mpio', 'tipo', 'latitud', 'longitud', 'geometry_wkt']

[INFO] ── Diagnóstico del DataFrame procesado ──
  → Shape      : 1,121 filas × 8 columnas
 

,cod_dpto,nom_dpto,cod_mpio,nom_mpio,tipo,latitud,longitud,geometry_wkt
0,05,ANTIOQUIA,05001,MEDELLÍN,MUNICIPIO,6.257590,-75.611031,POINT (-75.61103107 6.257590259)
1,05,ANTIOQUIA,05002,ABEJORRAL,MUNICIPIO,5.803728,-75.438474,POINT (-75.43847353 5.803728154)
2,05,ANTIOQUIA,05004,ABRIAQUÍ,MUNICIPIO,6.627569,-76.085978,POINT (-76.08597756 6.627569378)
3,05,ANTIOQUIA,05021,ALEJANDRÍA,MUNICIPIO,6.365534,-75.090597,POINT (-75.09059702 6.365534125)
4,05,ANTIOQUIA,05030,AMAGÁ,MUNICIPIO,6.032922,-75.708003,POINT (-75.7080031 6.032921994)



  Muestra (deptos especiales — Bogotá, Amazonas, San Andrés):


,cod_dpto,nom_dpto,cod_mpio,nom_mpio,tipo,latitud,longitud,geometry_wkt
148,11,"BOGOTÁ, D.C.",11001,"BOGOTÁ, D.C.",MUNICIPIO,4.316108,-74.181073,POINT (-74.1810727 4.316107698)
1086,88,"ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANT...",88001,SAN ANDRÉS,ISLA,12.543115,-81.717624,POINT (-81.71762382 12.54311512)
1087,88,"ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANT...",88564,PROVIDENCIA,MUNICIPIO,13.351110,-81.373885,POINT (-81.37388549 13.3511096)
1088,91,AMAZONAS,91001,LETICIA,MUNICIPIO,-3.530059,-70.045137,POINT (-70.04513691 -3.530058784)
1089,91,AMAZONAS,91263,EL ENCANTO,ÁREA NO MUNICIPALIZADA,-1.997235,-72.723616,POINT (-72.72361566 -1.997234795)
1090,91,AMAZONAS,91405,LA CHORRERA,ÁREA NO MUNICIPALIZADA,-1.509800,-72.444430,POINT (-72.44442966 -1.509799605)


[INFO] Exportando a Parquet: /kaggle/working/divipola_geo_procesado.parquet
  → Guardado | Filas: 1,121 | Columnas: 8

[OK] Pipeline completado.
